In [1]:
!apt-get -qq update
!apt-get -qq install -y default-jdk
!pip -q install cryptography pandas openpyxl scikit-learn

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package at-spi2-common.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../00-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../01-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../02-libxtst6_2%3a1.2.3-1.1build1_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1.1build1) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../03-session-migration_0.3.9build1_amd64.deb ...
Unpacking session-migration (0.3.9build1) ...
Selecting previously unse

In [2]:
import subprocess
import sys
import pandas as pd
import cryptography

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("Cryptography:", cryptography.__version__)

result = subprocess.run(
    ["keytool", "-help"],
    capture_output=True,
    text=True
)

print("Keytool available:", result.returncode == 0)

Python: 3.13.16
Pandas: 2.2.3
Cryptography: 50.0.2
Keytool available: True


In [3]:
import subprocess
import os

test_password = "TestPass123!"
test_file = "demo.jks"

# Remove an old test file if it exists
if os.path.exists(test_file):
    os.remove(test_file)

command = [
    "keytool",
    "-genkeypair",
    "-alias", "demo-cert",
    "-keyalg", "RSA",
    "-keysize", "2048",
    "-validity", "365",
    "-dname", "CN=Demo Certificate, OU=AI Lab, O=Test, C=US",
    "-keystore", test_file,
    "-storetype", "JKS",
    "-storepass", test_password,
    "-keypass", test_password
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("JKS file created successfully.")
    print("File name:", test_file)
    print("File exists:", os.path.exists(test_file))
else:
    print("Error:", result.stderr)

JKS file created successfully.
File name: demo.jks
File exists: True


In [4]:
result = subprocess.run(
    [
        "keytool",
        "-list",
        "-v",
        "-keystore", "demo.jks",
        "-storepass", test_password
    ],
    capture_output=True,
    text=True
)

print(result.stdout[:5000])

if result.returncode != 0:
    print(result.stderr)

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Keystore type: JKS
Keystore provider: SUN

Your keystore contains 1 entry

Alias name: demo-cert
Creation date: Oct 18, 2026
Entry type: PrivateKeyEntry
Certificate chain length: 1
Certificate[1]:
Owner: CN=Demo Certificate, OU=AI Lab, O=Test, C=US
Issuer: CN=Demo Certificate, OU=AI Lab, O=Test, C=US
Serial number: 7cc6a895a024ca1
Valid from: Sun Oct 18 21:03:23 UTC 2026 until: Mon Oct 18 21:03:23 UTC 2027
Certificate fingerprints:
	 SHA1: C3:B9:25:D5:17:87:10:E4:46:49:A3:6E:61:80:B0:A4:7B:2E:C3:BC
	 SHA256: AC:68:CE:D4:CF:71:22:22:ED:ED:DF:BF:92:4D:D3:9B:05:E2:51:8E:3A:50:86:F3:43:0C:52:38:96:7A:C3:D0
Signature algorithm name: SHA384withRSA
Subject Public Key Al

In [6]:

def extract_certificates(filename, password):

    # First, list the keystore entries in verbose format.
    result = subprocess.run(
        [
            "keytool",
            "-list",
            "-v",
            "-keystore", filename,
            "-storepass", password
        ],
        capture_output=True,
        text=True,
        errors="replace"
    )

    if result.returncode != 0:
        raise RuntimeError(
            result.stderr.strip() or result.stdout.strip()
        )

    # Split the output into alias sections.
    sections = re.split(
        r"(?m)^Alias name:\s*",
        result.stdout
    )[1:]

    records = []

    for section in sections:
        alias = section.splitlines()[0].strip()

        # Export each certificate separately in PEM format.
        pem_result = subprocess.run(
            [
                "keytool",
                "-exportcert",
                "-rfc",
                "-alias", alias,
                "-keystore", filename,
                "-storepass", password
            ],
            capture_output=True,
            text=True,
            errors="replace"
        )

        if pem_result.returncode != 0:
            print(
                f"Could not export certificate for alias {alias}:",
                pem_result.stderr.strip()
            )
            continue

        pem_blocks = re.findall(
            r"-----BEGIN CERTIFICATE-----.*?-----END CERTIFICATE-----",
            pem_result.stdout,
            flags=re.DOTALL
        )

        for index, pem in enumerate(pem_blocks, start=1):
            cert = x509.load_pem_x509_certificate(
                pem.encode("ascii")
            )

            key_algorithm, key_size = get_public_key_info(
                cert.public_key()
            )

            now = datetime.now(timezone.utc)
            expiry = cert.not_valid_after_utc
            start = cert.not_valid_before_utc
            days_left = (expiry - now).days

            if now < start:
                status = "Not Yet Valid"
            elif now > expiry:
                status = "Expired"
            elif days_left <= 30:
                status = "Expires Soon"
            else:
                status = "Valid"

            signature_hash = cert.signature_hash_algorithm

            records.append({
                "File": filename,
                "Alias": alias,
                "Certificate Number": index,
                "Subject": cert.subject.rfc4514_string(),
                "Issuer": cert.issuer.rfc4514_string(),
                "Serial Number": hex(cert.serial_number),
                "Key Algorithm": key_algorithm,
                "Key Size": key_size,
                "Signature Hash": (
                    signature_hash.name
                    if signature_hash else "Not Applicable"
                ),
                "Valid From UTC": start.isoformat(),
                "Expires At UTC": expiry.isoformat(),
                "Days Remaining": days_left,
                "Status": status,
                "SHA256 Fingerprint": cert.fingerprint(
                    hashes.SHA256()
                ).hex().upper()
            })

    return records


In [10]:
records = extract_certificates(
    "demo.jks",
    "TestPass123!"
)

print("Number of certificates:", len(records))

if records:
    print("\nFirst extracted record:")
    print(records[0])
else:
    print("No certificates found.")

Number of certificates: 1

First extracted record:
{'File': 'demo.jks', 'Alias': 'demo-cert', 'Certificate Number': 1, 'Subject': 'CN=Demo Certificate,OU=AI Lab,O=Test,C=US', 'Issuer': 'CN=Demo Certificate,OU=AI Lab,O=Test,C=US', 'Serial Number': '0x7cc6a895a024ca1', 'Key Algorithm': 'RSA', 'Key Size': 2048, 'Signature Hash': 'sha384', 'Valid From UTC': '2026-10-18T21:03:23+00:00', 'Expires At UTC': '2027-10-18T21:03:23+00:00', 'Days Remaining': 364, 'Status': 'Valid', 'SHA256 Fingerprint': 'AC68CED4CF712222EDEDDFBF924DD39B05E2518E3A5086F3430C5238967AC3D0'}


In [11]:
import pandas as pd

df = pd.DataFrame(records)

print("Total certificates:", len(df))
print("Total columns:", len(df.columns))

display(df)

Total certificates: 1
Total columns: 14


,File,Alias,Certificate Number,Subject,Issuer,Serial Number,Key Algorithm,Key Size,Signature Hash,Valid From UTC,Expires At UTC,Days Remaining,Status,SHA256 Fingerprint
0,demo.jks,demo-cert,1,"CN=Demo Certificate,OU=AI Lab,O=Test,C=US","CN=Demo Certificate,OU=AI Lab,O=Test,C=US",0x7cc6a895a024ca1,RSA,2048,sha384,2026-10-18T21:03:23+00:00,2027-10-18T21:03:23+00:00,364,Valid,AC68CED4CF712222EDEDDFBF924DD39B05E2518E3A5086...


In [12]:
import pandas as pd

if df.empty:
    print("No certificates available.")
else:
    df["Duplicate Fingerprint"] = df[
        "SHA256 Fingerprint"
    ].duplicated(keep=False)

    def analyze_certificate(row):
        findings = []

        if row["Status"] == "Expired":
            findings.append("Expired certificate")
        elif row["Status"] == "Expires Soon":
            findings.append("Expires within 30 days")
        elif row["Status"] == "Not Yet Valid":
            findings.append("Not yet valid")

        algorithm = str(row["Key Algorithm"]).upper()
        key_size = row["Key Size"]

        if pd.notna(key_size):
            if algorithm == "RSA" and key_size < 2048:
                findings.append("RSA key below 2048 bits")
            elif algorithm == "DSA" and key_size < 2048:
                findings.append("DSA key below 2048 bits")
            elif algorithm == "EC" and key_size < 224:
                findings.append("EC key below 224 bits")

        signature_hash = str(row["Signature Hash"]).upper()

        if signature_hash in {"MD5", "SHA1"}:
            findings.append("Legacy signature hash")

        if row["Duplicate Fingerprint"]:
            findings.append("Duplicate fingerprint")

        return "; ".join(findings) if findings else "No rule triggered"

    df["Analysis Findings"] = df.apply(
        analyze_certificate, axis=1
    )

    df["Needs Review"] = (
        df["Analysis Findings"] != "No rule triggered"
    )

    display(df[[
        "File",
        "Alias",
        "Key Algorithm",
        "Key Size",
        "Days Remaining",
        "Status",
        "Duplicate Fingerprint",
        "Analysis Findings",
        "Needs Review"
    ]])

,File,Alias,Key Algorithm,Key Size,Days Remaining,Status,Duplicate Fingerprint,Analysis Findings,Needs Review
0,demo.jks,demo-cert,RSA,2048,364,Valid,False,No rule triggered,False


In [13]:
if not df.empty:
    print("===== JKS ANALYSIS SUMMARY =====")
    print("Total certificates:", len(df))
    print("Expired:", (df["Status"] == "Expired").sum())
    print("Expiring soon:", (df["Status"] == "Expires Soon").sum())
    print("Not yet valid:", (df["Status"] == "Not Yet Valid").sum())
    print("Duplicate fingerprints:", df["Duplicate Fingerprint"].sum())
    print("Needs review:", df["Needs Review"].sum())

===== JKS ANALYSIS SUMMARY =====
Total certificates: 1
Expired: 0
Expiring soon: 0
Not yet valid: 0
Duplicate fingerprints: 0
Needs review: 0


In [14]:
from google.colab import files

if not df.empty:
    df.to_excel(
        "jks_certificate_report.xlsx",
        index=False
    )

    df.to_csv(
        "jks_certificate_report.csv",
        index=False,
        encoding="utf-8-sig"
    )

    df[df["Needs Review"]].to_excel(
        "jks_security_review.xlsx",
        index=False
    )

    print("Reports created successfully.")

    files.download("jks_certificate_report.xlsx")
    files.download("jks_certificate_report.csv")
    files.download("jks_security_review.xlsx")
else:
    print("No data to export.")

Reports created successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>